In [1]:
# Install ADK and LiteLLM
!pip install google-adk -q
!pip install google-adk[extensions] -q
!pip install litellm -q
!pip install nest_asyncio

print("dependencies installed...")

dependencies installed...


In [2]:
import os
from getpass import getpass

# Get inputs
PROJECT_ID = getpass("Enter your GCP project id: ")
GEMINI_API_KEY = getpass("Enter your Google Gemini API key: ")

# Set environment variables so LiteLLM and your functions automatically find them
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Credentials loaded successfully into environment!")

Enter your GCP project id: ··········
Enter your Google Maps API key: ··········
Enter your Google Gemini API key: ··········
Credentials loaded successfully into environment!


In [9]:
import os
import asyncio
import logging
import warnings
import nest_asyncio
from typing import Dict, Any

# Suppress ADK agent deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="google.adk")

from google.adk.agents import Agent, SequentialAgent, LoopAgent
from google.adk.tools import ToolContext, AgentTool, google_search
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.genai.types import Content, Part

# Apply nest_asyncio for Jupyter / Colab event loops
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO)

MODEL_NAME = "gemini-3.6-flash"
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=5)

# ============================================================================
# 1. State tracking function
# ============================================================================

def append_to_state(
    tool_context: ToolContext, field: str, response: str
) -> Dict[str, str]:
    """Appends new text output to an existing state list key in tool context."""
    existing_state = tool_context.state.get(field, [])
    if isinstance(existing_state, str):
        existing_state = [existing_state]
    tool_context.state[field] = existing_state + [response]
    logging.info(f"[Added to state field: '{field}'] {response[:100]}...")
    return {"status": "success"}


# Dedicated sub-agent isolating google_search
search_sub_agent = Agent(
    name="search_sub_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Performs live web searches using Google Search.",
    instruction="Use `google_search` to find up-to-date facts and verify statements.",
    tools=[google_search],
)

# ============================================================================
# Specialist Workflow Agents
# ============================================================================

# Initial Answer Draft
draft_researcher = Agent(
    name="draft_researcher",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Researches and drafts an initial comprehensive answer to the user query.",
    instruction="""
    USER_QUERY: { PROMPT? }

    INSTRUCTIONS:
    - Analyze the USER_QUERY.
    - Call `search_sub_agent` to search the web for verified facts, evidence, or current data.
    - Draft a thorough, clear initial response addressing all aspects of the query.
    - Call `append_to_state` to store your draft in the 'ANSWER_DRAFT' state field.
    """,
    tools=[AgentTool(agent=search_sub_agent), append_to_state],
)

# Verification & Refinement Loop
fact_verifier = Agent(
    name="fact_verifier",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Fact-checks the current answer draft against external search sources.",
    instruction="""
    USER_QUERY: { PROMPT? }
    ANSWER_DRAFT: { ANSWER_DRAFT? }

    INSTRUCTIONS:
    - Review the latest draft in ANSWER_DRAFT against the original USER_QUERY.
    - Use `search_sub_agent` to independently verify key facts, dates, statistics, or logical claims.
    - If inaccuracies, omissions, or unverified claims are found, call `append_to_state` to store specific corrections in 'VERIFICATION_NOTES'.
    - If the answer is completely accurate, comprehensive, and ready, output '[APPROVED]'.
    """,
    tools=[AgentTool(agent=search_sub_agent), append_to_state],
)

answer_refiner = Agent(
    name="answer_refiner",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Refines and rewrites the answer based on verification feedback.",
    instruction="""
    USER_QUERY: { PROMPT? }
    ANSWER_DRAFT: { ANSWER_DRAFT? }
    VERIFICATION_NOTES: { VERIFICATION_NOTES? }

    INSTRUCTIONS:
    - Revise and improve the ANSWER_DRAFT incorporating the corrections/additions from VERIFICATION_NOTES.
    - Ensure tone is objective, clear, and well-structured.
    - Call `append_to_state` to store the updated version back into the 'ANSWER_DRAFT' state field.
    """,
    tools=[append_to_state],
)

verification_loop = LoopAgent(
    name="verification_loop",
    description="Iteratively verifies and refines the draft until approved.",
    sub_agents=[fact_verifier, answer_refiner],
    max_iterations=1,
)

# ============================================================================
# Pipeline Assembly & Root Agent
# ============================================================================

qa_verification_team = SequentialAgent(
    name="qa_verification_team",
    description="Executes drafting, fact verification loop, and final refinement.",
    sub_agents=[
        draft_researcher,    # Step 1: Initial research & draft
        verification_loop,   # Step 2: Loop until fact-checker approves
    ],
)

root_agent = Agent(
    name="root_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Receives any user question and routes it through the verification pipeline.",
    instruction="""
    INSTRUCTIONS:
    - Receive the user question.
    - Save the query into the state key 'PROMPT' using `append_to_state`.
    - Transfer execution to 'qa_verification_team'.
    """,
    tools=[append_to_state],
    sub_agents=[qa_verification_team],
)



/tmp/ipykernel_126351/1439870483.py:109: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  verification_loop = LoopAgent(
/tmp/ipykernel_126351/1439870483.py:120: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  qa_verification_team = SequentialAgent(


In [10]:
# ============================================================================
#  Interactive Execution Runner
# ============================================================================

APP_NAME = "verified_qa_app"
session_service = InMemorySessionService()

async def get_or_create_session(user_id: str, session_id: str):
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=user_id, session_id=session_id
    )
    if not session:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=user_id, session_id=session_id
        )
    return session

def run_interactive_qa_assistant(
    user_id: str = "qa_user",
    session_id: str = "session_qa_001"
):
    """Runs an interactive QA session that takes open-ended questions from the user."""

    print("\n" + "="*60)
    print("🧠 VERIFIED QUESTION & ANSWER AGENT SYSTEM")
    print("="*60)
    print("Ask any factual or complex question.")
    print("Our multi-agent pipeline will draft, fact-check via search,")
    print("and refine the response prior to answering.")
    print("-" * 60)

    user_query = input("👉 Enter your question: ").strip()

    if not user_query:
        print("⚠️ No query entered. Defaulting to a test question.")
        user_query = "Why is the sky blue?"

    print(f"\n🚀 Processing & Verifying Answer for: '{user_query}'...\n" + "="*60)

    loop = asyncio.get_event_loop()
    loop.run_until_complete(
        get_or_create_session(user_id=user_id, session_id=session_id)
    )

    runner = Runner(
        app_name=APP_NAME,
        agent=root_agent,
        session_service=session_service,
    )

    formatted_message = Content(
        role="user",
        parts=[Part.from_text(text=user_query)]
    )

    event_stream = runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=formatted_message,
    )

    for event in event_stream:
        if hasattr(event, "author") and event.author:
            print(f"\n🤖 [{event.author}]:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(part.text)

    print("\n" + "="*60)
    print("✅ Verified Answer Complete!")

# Run interactive session
run_interactive_qa_assistant()


🧠 VERIFIED QUESTION & ANSWER AGENT SYSTEM
Ask any factual or complex question.
Our multi-agent pipeline will draft, fact-check via search,
and refine the response prior to answering.
------------------------------------------------------------
👉 Enter your question: Why is the grass green?

🚀 Processing & Verifying Answer for: 'Why is the grass green?'...

🤖 [root_agent]:

🤖 [root_agent]:

🤖 [root_agent]:

🤖 [root_agent]:

🤖 [draft_researcher]:

🤖 [draft_researcher]:

🤖 [draft_researcher]:

🤖 [draft_researcher]:

🤖 [draft_researcher]:
Grass appears green primarily because of a natural pigment called **chlorophyll** and the way it interacts with sunlight.

---

### 1. The Role of Chlorophyll and Photosynthesis
Inside grass cells are microscopic structures called **chloroplasts**, which contain **chlorophyll**. Chlorophyll is the key molecule that plants use for **photosynthesis**—the chemical process by which plants convert sunlight, water, and carbon dioxide into energy (glucose) and 